In [ ]:
def int_to_poly(p: int):
    poly = ""
    for i in range(p.bit_length()):
        bit = (p >> i) & 1
        if bit:
            if i == 0:
                add = "1"
            elif i == 1:
                add = "x"
            else:
                add = f"x^{i}"

            if poly == "":
                poly = add
            else:
                poly += f" + {add}"

    return poly


print(int_to_poly(19))

In [ ]:
import sys
from pathlib import Path

path_root = Path.cwd().parent

if str(path_root) not in sys.path:
    sys.path.append(str(path_root))

In [ ]:
# TEST: does multiply_polynomials_mod_l correctly multiply polynomials with coefficients mod 2 and exponents mod l
from code_construction.code_construction import CodeConstructor, CSSCode

p1 = 417  # 110100001 -> 1 + x^5 + x^7 + x^8
p2 = 9  # 1001 -> 1 + x^3

l = 9

# p1 * p2 = 1 + x^5 + x^7 + x^8 + x^3 + x^8 + x^10 + x^11 = 1 + x^5 + x^7 + x^8 + x^3 + x^8 + x + x^2 = 1 + x + x^2 + x^3 + x^5 + x^7

p1p2 = CodeConstructor.multiply_polynomials_mod_l(p1, p2, l)
print(int_to_poly(p1))
print(int_to_poly(p2))
print(int_to_poly(p1p2))

assert int_to_poly(p1p2) == "1 + x + x^2 + x^3 + x^5 + x^7"

print("Test Passed!")

In [ ]:
# TEST: does gx_mask_to_bin correctly generate the g(x) polynomial in int form given a list of factors and a bitmask
fs = [
    5,  # 101 -> 1+x^2
    49,  # 110001 -> 1+x^4+x^5
    17,  # 10001 -> 1+x^4
]


gx_mask = 6  # 110 -> (1+x^4+x^5)(1+x^4) = 1 + x^4 + x^5 + x^4 + x^8 + x^9 = 1 + x^5 + x^8 + 1 = x^5 + x^8

gx_bin = CodeConstructor.gx_mask_to_bin(gx_mask, fs, l)

for f in fs:
    print(int_to_poly(f))

print(int_to_poly(gx_bin))

assert int_to_poly(gx_bin) == "x^5 + x^8"

print("Test Passed!")

In [ ]:
# TEST: are randomly generated GB codes valid
from BOplayground import Get_new_points_function


# this is the same as multiply_polynomials_mod_l but without the mod l
def binary_poly_mul(p1, p2):
    res = 0
    for i in range(p2.bit_length()):
        if (p2 >> i) & 1:
            res ^= p1 << i
    return res


para_dict = {"l": l}
code_class = "gb"
target_weight = 3
desired_k = 12

cc = CodeConstructor(method=code_class, para_dict=para_dict)


gnp = Get_new_points_function(
    method=code_class, code_constructor=cc, density=target_weight, desired_k=desired_k
)

gnp.set_gx_mask()


x_exp_l_minus_1 = 1 + (1 << l)  # x^l - 1 = x^l + 1 in binary ring

new_points = gnp.get_new_points_function(5)

for point in new_points:
    gx_mask, qa, qb = [int(x) for x in point[:3]]
    fs = [int(x) for x in point[3:]]
    prod_fs = 1
    for f in fs:
        prod_fs = binary_poly_mul(prod_fs, f)

    assert prod_fs == x_exp_l_minus_1

    gx_bin = CodeConstructor.gx_mask_to_bin(gx_mask, fs, l)
    assert (
        CodeConstructor.multiply_polynomials_mod_l(gx_bin, qa, l).bit_count()
        == target_weight
    )  # LHS: a(x)
    assert (
        CodeConstructor.multiply_polynomials_mod_l(gx_bin, qb, l).bit_count()
        == target_weight
    )  # LHD: b(x)

print("Test Passed!")

In [ ]:
small_l = 3
print("l=3")

# factors of x^3 - 1 in the ring should be (x + 1)(x^2 + x + 1)
factors = gnp._get_irreducible_factors(small_l)
for f in factors:
    print(f"{int_to_poly(f)} ({f})")

print("\n")
print("l=9")

# factors of x^3 - 1 in the ring should be (x + 1)(x^2 + x + 1)
factors = gnp._get_irreducible_factors(l)
for f in factors:
    print(f"{int_to_poly(f)} ({f})")

In [ ]:
from sympy import symbols, factor, expand, GF


def factor_poly(poly):
    factors = factor(poly, domain=GF(2))
    return factors


x = symbols("x")
print(factor_poly(1 + x + x**5))

print(expand((1 + x**2 + x**3) * (1 + x), modulus=2))
print(expand((1 + x + x**2) * (1 + x), modulus=2))

In [ ]:
# TEST: is the CSS code generated by CodeConstructor correct for a given GB internal representation
import numpy as np

a = 23  # 000010111 -> 1 + x + x^2 + x^4
b = 9  # 000001001 -> 1 + x^3

qa = 13  # 1101 -> 1 + x^2 + x^3
qb = 7  # 111 -> 1 + x + x^2
gx_mask = 1  # 001, gx_bin=3
fs = [3, 7, 73]  # from l=9 code above
gx_bin = CodeConstructor.gx_mask_to_bin(gx_mask, fs, l)

assert CodeConstructor.multiply_polynomials_mod_l(gx_bin, qa, l) == a
assert CodeConstructor.multiply_polynomials_mod_l(gx_bin, qb, l) == b

x = [gx_mask, qa, qb] + fs

a_lsb_first = [0, 0, 0, 0, 1, 0, 1, 1, 1][
    ::-1
]  # bit order is flipped for A&B matrices (least significant bit on the left)
b_lsb_first = [0, 0, 0, 0, 0, 1, 0, 0, 1][::-1]

expected_A = np.array(
    [
        a_lsb_first,
        np.roll(a_lsb_first, 1),
        np.roll(a_lsb_first, 2),
        np.roll(a_lsb_first, 3),
        np.roll(a_lsb_first, 4),
        np.roll(a_lsb_first, 5),
        np.roll(a_lsb_first, 6),
        np.roll(a_lsb_first, 7),
        np.roll(a_lsb_first, 8),
    ]
)

expected_B = np.array(
    [
        b_lsb_first,
        np.roll(b_lsb_first, 1),
        np.roll(b_lsb_first, 2),
        np.roll(b_lsb_first, 3),
        np.roll(b_lsb_first, 4),
        np.roll(b_lsb_first, 5),
        np.roll(b_lsb_first, 6),
        np.roll(b_lsb_first, 7),
        np.roll(b_lsb_first, 8),
    ]
)

expected_hx = np.hstack((expected_A, expected_B))

expected_hz = np.hstack((expected_B.T, expected_A.T))

css = cc.construct(x)

assert isinstance(css, CSSCode)
assert (((css.hx @ css.hz.T) % 2) == 0).all()  # CSS parity check constraint

assert np.array_equal(css.hx, expected_hx)
assert np.array_equal(css.hz, expected_hz)

print("Test Passed!")

In [ ]:
# TEST: are generated neighbours of a GB code valid, do they have the same g(x) and have one q the same, and one different
from BOplayground import HillClimbing
import torch

# gx_mask = None
# qa = None
# qb = None

# l = None

# fs = []
qa = 3  # 11 -> 1 + x
qb = 5  # 111 -> 1 + x + x^2
gx_mask = 3  # 001

fs = [3, 7, 73]  # from l=9 code above
gx_bin = CodeConstructor.gx_mask_to_bin(gx_mask, fs, l)


x = [gx_mask, qa, qb] + fs

# CodeConstructor.multiply_polynomials_mod_l(gx_bin, n[1], l).bit_count()

# gx_bin = CodeConstructor.gx_mask_to_bin(gx_mask, fs, l)
target_weight2 = 4

gnp2 = Get_new_points_function(
    method=code_class, code_constructor=cc, density=target_weight2, desired_k=desired_k
)

gnp2.set_gx_mask()

hc = HillClimbing(
    next_points_num=1,
    gnp=gnp2.get_new_points_function,
    acquisition=None,
    l=l,
    target_row_weight=target_weight2,
)

x_tensor = torch.tensor(x, dtype=torch.float32)

x_neighbours = hc.swap_factors(x_tensor)
assert len(x_neighbours) > 0
print(len(x_neighbours))

for n in x_neighbours:
    n = n.to(torch.long).tolist()
    assert n[0] == gx_mask
    assert (n[1] == qa) or (n[2] == qb)
    assert np.array_equal(n[3:], fs)

    assert (
        CodeConstructor.multiply_polynomials_mod_l(gx_bin, n[1], l).bit_count()
        == target_weight2
    )
    assert (
        CodeConstructor.multiply_polynomials_mod_l(gx_bin, n[2], l).bit_count()
        == target_weight2
    )
    assert n != x

    cc.construct(n)  # CSSCode has built in checks to make sure the code is valid

print("Test Passed!")